## Transfer Learning

In [4]:
import datetime
import keras

from keras.datasets import mnist
from keras.models import Sequential
from keras import backend as K
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Dense, Dropout, Activation, Flatten

now = datetime.datetime.now

In [5]:
# params
batch_size = 128
n_classes = 5
epochs = 5

img_rows, img_cols = 28, 28
filters = 32
pool_size = 2
kernel_size = 3

if K.image_data_format() == 'channels_first':
    input_shapes = (1, img_rows, img_cols)
else:
    input_shapes = (img_rows, img_cols, 1)

In [8]:
(xtrain, ytrain), (xtest, ytest) = mnist.load_data()


xtrain_lt5 = xtrain[ytrain < 5]
ytrain_lt5 = ytrain[ytrain < 5]
xtest_lt5 = xtest[ytest < 5]
ytest_lt5 = ytest[ytest < 5]

xtrain_gte5 = xtrain[ytrain >= 5]
ytrain_gte5 = ytrain[ytrain >= 5] - 5
xtest_gte5 = xtest[ytest >= 5]
ytest_gte5 = ytest[ytest >= 5] - 5

In [12]:
# helper function

def _train_model(model, train, test, num_classes):

    xtrain = train[0].reshape((train[0].shape[0],) + input_shapes) 
    xtest = test[0].reshape((test[0].shape[0],) + input_shapes)

    xtrain = xtrain.astype('float32')
    xtest = xtest.astype('float32')

    xtrain /= 255
    xtest /= 255

    print('x_train shape:', xtrain.shape)
    print(xtrain.shape[0], 'train samples')
    print(xtest.shape[0], 'test samples')

    ytrain = keras.utils.to_categorical(train[1], num_classes)
    ytest = keras.utils.to_categorical(test[1], num_classes)

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adadelta', metrics=['accuracy']
    )

    t = now()

    model.fit(
        xtrain, ytrain, batch_size=batch_size,
        epochs=epochs, verbose=1, validation_data=(xtest, ytest)
    )

    print('Training time: %s' % (now() - t))
    
    score = model.evaluate(xtest, ytest, verbose=0)
    print('Test score:', score[0])
    print('Test accuracy:', score[1])

In [7]:
feature_layers = [
    Conv2D(filters, kernel_size, padding='valid', input_shape=input_shapes),
    Activation('relu'),
    Conv2D(filters, kernel_size=kernel_size),
    Activation('relu'),
    MaxPooling2D(pool_size),
    Dropout(0.25),
    Flatten()
]

classification_layers = [
    Dense(128),
    Activation('relu'),
    Dropout(0.5),
    Dense(n_classes),
    Activation('softmax')
]

model = Sequential(feature_layers + classification_layers)
model.summary()

c:\Users\LEI\Documents\deep-learning-ibm\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 26, 26, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 5)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 600,165 (2.29 MB)

 Trainable params: 600,165 (2.29 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
_train_model(
    model,
    (xtrain_gte5, ytrain_gte5),
    (xtest_gte5, ytest_gte5), n_classes
)

x_train shape: (29404, 28, 28, 1)
29404 train samples
4861 test samples
Epoch 1/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 15s 60ms/step - accuracy: 0.2345 - loss: 1.6024 - val_accuracy: 0.3409 - val_loss: 1.5878
Epoch 2/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 14s 62ms/step - accuracy: 0.2895 - loss: 1.5841 - val_accuracy: 0.4682 - val_loss: 1.5656
Epoch 3/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 15s 65ms/step - accuracy: 0.3508 - loss: 1.5629 - val_accuracy: 0.5641 - val_loss: 1.5415
Epoch 4/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 15s 63ms/step - accuracy: 0.4084 - loss: 1.5404 - val_accuracy: 0.6396 - val_loss: 1.5144
Epoch 5/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 14s 61ms/step - accuracy: 0.4707 - loss: 1.5141 - val_accuracy: 0.6951 - val_loss: 1.4835
Training time: 0:01:13.468330
Test score: 1.4835106134414673
Test accuracy: 0.6951244473457336


## Freezing Layers

In [14]:
for l in feature_layers:
    l.trainable = False

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 26, 26, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 5)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,800,497 (6.87 MB)

 Trainable params: 590,597 (2.25 MB)

 Non-trainable params: 9,568 (37.38 KB)

 Optimizer params: 1,200,332 (4.58 MB)

In [16]:
_train_model(
    model, (xtrain_lt5, ytrain_lt5), 
    (xtest_lt5, ytest_lt5), n_classes
)

x_train shape: (30596, 28, 28, 1)
30596 train samples
5139 test samples
Epoch 1/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 12s 46ms/step - accuracy: 0.3124 - loss: 1.5743 - val_accuracy: 0.4318 - val_loss: 1.5460
Epoch 2/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 8s 31ms/step - accuracy: 0.3937 - loss: 1.5346 - val_accuracy: 0.5472 - val_loss: 1.5016
Epoch 3/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - accuracy: 0.4747 - loss: 1.4926 - val_accuracy: 0.6606 - val_loss: 1.4577
Epoch 4/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.5377 - loss: 1.4526 - val_accuracy: 0.7428 - val_loss: 1.4142
Epoch 5/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.5953 - loss: 1.4129 - val_accuracy: 0.7852 - val_loss: 1.3712
Training time: 0:00:40.081086
Test score: 1.3712213039398193
Test accuracy: 0.7851722240447998
